# XML inspection: status and conversion to CSV

This Notebook first checks the status of the attempted downloads so that statistics can be reported in the thesis. Second, all available valid XMLs are converted into CSVs so that they are ready for the masking step.

In [1]:
import pandas as pd
import gzip
import json

## XML download status

In [2]:
download_status_file = "/scratch/project_2017974/data_collection/data/xml_download_status.txt"

downloaded = set()
failed = set()

def get_id(line: str):
    id_ = line.strip().split()[1]
    assert id_.startswith("W"), f"Problem parsing id from line {repr(line)}"
    return id_
    
with open(download_status_file) as file:
    for line in file:
        if line.startswith("#"):
            id_ = get_id(line)
            downloaded.add(id_)
        elif line.startswith("!"):
            id_ = get_id(line)
            failed.add(id_)

print("Downloaded:", len(downloaded))
print("Failed:", len(failed))
assert len(downloaded.intersection(failed)) == 0, "Same ID(s) found in downloaded and failed"

Downloaded: 5214
Failed: 1347


In [3]:
total = len(downloaded) + len(failed)
success_rate = len(downloaded) / total

print(f"Success rate: {success_rate:.2%}")

Success rate: 79.47%


In [4]:
metadata_file = "/scratch/project_2017974/data_collection/data/light-data/light-new-query-dedupe.jsonl.gz"

def get_status(id_: str):
    if id_ in downloaded:
        return "ok"
    elif id_ in failed:
        return "failed"
    return "pending"

data_with_xml = []
with gzip.open(metadata_file) as file:
    for line in file:
        obj = json.loads(line)
        has_xml = obj.get("has_xml")
        if has_xml != True:
            continue
        id_ = obj.get("id")
        new_obj = {
            "id": id_,
            "title": obj.get("title"),
            "year": obj.get("publication_year"),
            "topic": obj.get("topic"),
            "subfield": obj.get("subfield"),
            "field": obj.get("field"),
            "domain": obj.get("domain"),
            "license": obj.get("license"),
            "status": get_status(id_)
        }
        data_with_xml.append(new_obj)

print("Has XML:", len(data_with_xml))

Has XML: 135559


In [5]:
cat_cols = ("year", "topic", "subfield", "field", "domain", "license", "status")
dtypes = {cat: "category" for cat in cat_cols}
meta = pd.DataFrame(data_with_xml)
meta = meta.astype(dtypes)
meta.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 135559 entries, 0 to 135558
Data columns (total 9 columns):
 #   Column    Non-Null Count   Dtype   
---  ------    --------------   -----   
 0   id        135559 non-null  object  
 1   title     135559 non-null  object  
 2   year      135559 non-null  category
 3   topic     135558 non-null  category
 4   subfield  135558 non-null  category
 5   field     135558 non-null  category
 6   domain    135558 non-null  category
 7   license   92198 non-null   category
 8   status    135559 non-null  category
dtypes: category(7), object(2)
memory usage: 3.4+ MB


In [6]:
dl = meta[meta["status"] == "ok"]
dl

,id,title,year,topic,subfield,field,domain,license,status
85,W1585628326,Delivering the Smart Grid: Challenges for Auto...,2021,Smart Grid Energy Management,Electrical and Electronic Engineering,Engineering,Physical Sciences,NaN,ok
89,W1591705377,Advances in Artificial Intelligence: Are You S...,2017,"Computability, Logic, AI Algorithms",Computational Theory and Mathematics,Computer Science,Physical Sciences,NaN,ok
95,W1598176529,Methodologies for Developing Multi-Agent Systems,2020,Multi-Agent Systems and Negotiation,Artificial Intelligence,Computer Science,Physical Sciences,other-oa,ok
113,W169306565,The Impact of Artificial Intelligence on Society,2023,Impact of AI and Big Data on Business and Society,Management Science and Operations Research,Decision Sciences,Social Sciences,other-oa,ok
882,W2117592456,Unifying An Introduction To Artificial Intelli...,2020,Machine Learning and Data Classification,Artificial Intelligence,Computer Science,Physical Sciences,NaN,ok
...,...,...,...,...,...,...,...,...,...
134087,W7136068103,Triple-Gate Actuation: A Hardware-Isolated Saf...,2026,Robot Manipulation and Learning,Control and Systems Engineering,Engineering,Physical Sciences,cc-by,ok
134091,W7136089029,SuperLocalMemory V3: Information-Geometric Fou...,2026,Topological and Geometric Data Analysis,Computational Theory and Mathematics,Computer Science,Physical Sciences,cc-by,ok
134301,W7138989979,Persona Framing in Human–LLM Interaction,2026,Topic Modeling,Artificial Intelligence,Computer Science,Physical Sciences,cc-by,ok
134367,W7139972609,Layer Mismatch: A Structural Account of Recomp...,2026,Distributed systems and fault tolerance,Computer Networks and Communications,Computer Science,Physical Sciences,NaN,ok


In [7]:
for col in ("year", "field", "domain"):
    print(dl[col].value_counts()[lambda x: x > 0])
    print()

year
2021    645
2022    611
2020    584
2025    572
2024    517
2023    503
2019    451
2018    447
2017    390
2016    333
2026    161
Name: count, dtype: int64

field
Computer Science                                1204
Medicine                                        1042
Social Sciences                                  532
Biochemistry, Genetics and Molecular Biology     532
Neuroscience                                     351
Agricultural and Biological Sciences             280
Engineering                                      252
Business, Management and Accounting              209
Psychology                                       191
Decision Sciences                                174
Health Professions                               106
Economics, Econometrics and Finance               79
Environmental Science                             73
Arts and Humanities                               36
Immunology and Microbiology                       34
Physics and Astronomy              

In [8]:
fail = meta[meta["status"] == "failed"]
fail

,id,title,year,topic,subfield,field,domain,license,status
1262,W2256456600,State-of-the-Art and Future Directions,2024,Explainable Artificial Intelligence (XAI),Artificial Intelligence,Computer Science,Physical Sciences,NaN,failed
1314,W2294511912,5G-Enabled Tactile Internet,2016,IoT and Edge/Fog Computing,Computer Networks and Communications,Computer Science,Physical Sciences,other-oa,failed
1403,W2334816998,"Fullrmc, a rigid body reverse monte carlo mode...",2016,Machine Learning in Materials Science,Materials Chemistry,Materials Science,Physical Sciences,NaN,failed
1424,W2345190899,Two Efficient Lattice Rescoring Methods Using ...,2016,Speech Recognition and Synthesis,Artificial Intelligence,Computer Science,Physical Sciences,NaN,failed
1437,W2397055186,Backstepping approach to a class of hierarchic...,2016,Distributed Control Multi-Agent Systems,Computer Networks and Communications,Computer Science,Physical Sciences,NaN,failed
...,...,...,...,...,...,...,...,...,...
134872,W7140878526,Zero-shot performance of selected large langua...,2026,Artificial Intelligence in Healthcare and Educ...,Health Informatics,Medicine,Health Sciences,cc-by-nc-nd,failed
135273,W7149371371,It Was Always the Words: Why machines had to l...,2026,Embodied and Extended Cognition,Cognitive Neuroscience,Neuroscience,Life Sciences,cc-by,failed
135377,W7151611085,An Assessment of GPT-3.5 and GPT-4.0 Responses...,2026,Artificial Intelligence in Healthcare and Educ...,Health Informatics,Medicine,Health Sciences,cc-by,failed
135401,W7152131640,Navigating the ethical landscape of scholarly ...,2026,Authorship Attribution and Profiling,Artificial Intelligence,Computer Science,Physical Sciences,cc-by,failed


In [9]:
for col in ("year", "field", "domain"):
    print(fail[col].value_counts()[lambda x: x > 0])
    print()

year
2025    305
2026    297
2024    270
2023    230
2022     46
2020     37
2021     36
2017     36
2016     34
2019     33
2018     23
Name: count, dtype: int64

field
Medicine                                        321
Computer Science                                274
Social Sciences                                 155
Biochemistry, Genetics and Molecular Biology    116
Neuroscience                                     94
Agricultural and Biological Sciences             61
Psychology                                       61
Engineering                                      58
Business, Management and Accounting              54
Decision Sciences                                39
Economics, Econometrics and Finance              25
Health Professions                               21
Immunology and Microbiology                      13
Arts and Humanities                              12
Environmental Science                             9
Materials Science                                 

In [10]:
counts = (
    meta[meta["status"].isin(["ok", "failed"])]
    .groupby(["status", "year"], observed=True)
    .size()
)
percentages = counts / counts.groupby("year", observed=True).transform("sum") * 100
percentages.unstack("status")

status,failed,ok
year,,
2016,9.264305,90.735695
2017,8.450704,91.549296
2018,4.893617,95.106383
2019,6.818182,93.181818
2020,5.958132,94.041868
2021,5.286344,94.713656
2022,7.001522,92.998478
2023,31.377899,68.622101
2024,34.307497,65.692503


In [11]:
counts = (
    meta[meta["status"].isin(["ok", "failed"])]
    .groupby(["status", "domain"], observed=True)
    .size()
)
percentages = counts / counts.groupby("domain", observed=True).transform("sum") * 100
percentages.unstack("status")

status,failed,ok
domain,,
Health Sciences,22.890778,77.109222
Life Sciences,19.187209,80.812791
Physical Sciences,18.482688,81.517312
Social Sciences,22.080408,77.919592


In [12]:
pend = meta[meta["status"] == "pending"]
pend

,id,title,year,topic,subfield,field,domain,license,status
0,W101656788,One-Match and All-Match Categories for Keyword...,2010,AI in Service Interactions,Artificial Intelligence,Computer Science,Physical Sciences,cc-by,pending
1,W1147321658,HClass: Automatic classification tool for heal...,2015,Machine Learning and Data Classification,Artificial Intelligence,Computer Science,Physical Sciences,NaN,pending
2,W117380508,From Algorithms to (Sub-)Symbolic Inferences i...,2006,Multi-Agent Systems and Negotiation,Artificial Intelligence,Computer Science,Physical Sciences,cc-by-nc,pending
3,W1173882893,Artificial intelligence: Immediate promises an...,1985,Big Data and Business Intelligence,Management Information Systems,"Business, Management and Accounting",Social Sciences,NaN,pending
4,W128729319,Provably Bounded-Optimal Agents,1995,"Computability, Logic, AI Algorithms",Computational Theory and Mathematics,Computer Science,Physical Sciences,cc-by,pending
...,...,...,...,...,...,...,...,...,...
135554,W94253779,Book review: Theoretical Aspects of Reasoning ...,1987,AI-based Problem Solving and Planning,Artificial Intelligence,Computer Science,Physical Sciences,NaN,pending
135555,W942635084,TWITTER USERS POPULARITY ESTIMATION USING EXPE...,2014,Complex Network Analysis Techniques,Statistical and Nonlinear Physics,Physics and Astronomy,Physical Sciences,cc-by,pending
135556,W97022837,Composition of zeolites of the natrolite group...,1965,Clay minerals and soil interactions,Biomaterials,Materials Science,Physical Sciences,NaN,pending
135557,W979034868,Europski uzori i hrvatski jezični priručnici u...,2015,Lexicography and Language Studies,Language and Linguistics,Arts and Humanities,Social Sciences,cc-by,pending


In [13]:
len(pend[pend["year"] == 2026])

6399

In [14]:
for col in ("year", "field", "domain"):
    print(pend[col].value_counts()[lambda x: x > 0])

year
2025    46789
2024    26484
2022    11882
2023     9896
2021     8254
        ...  
1911        1
1912        1
1914        1
1915        1
1918        1
Name: count, Length: 113, dtype: int64
field
Computer Science                                46348
Medicine                                        16544
Engineering                                     15207
Social Sciences                                 12331
Business, Management and Accounting              7438
Agricultural and Biological Sciences             3798
Psychology                                       3742
Decision Sciences                                3718
Environmental Science                            3302
Biochemistry, Genetics and Molecular Biology     3086
Economics, Econometrics and Finance              2855
Health Professions                               2406
Neuroscience                                     1860
Arts and Humanities                              1533
Physics and Astronomy                   

## Collect all XML files

In [ ]:
from bs4 import BeautifulSoup, NavigableString
from pathlib import Path
import re
import copy
from tqdm import tqdm

In [8]:
XML_ROOT = "/scratch/project_2017974/data_collection/data/xml-data"
all_xml_files = sorted(Path(XML_ROOT).glob("2026*/*.grobid.xml.gz"))

In [9]:
len(set(all_xml_files))

6076

In [10]:
xml_ids = [f.name.split(".")[0] for f in all_xml_files]

In [11]:
len(set(xml_ids).intersection(downloaded))

5214

Trying to remember why the counts would diverge here; the list was updated at some point - had I initially downloaded fulltexts that were not included in the files downloaded from OpenAlex with the new query...? Maybe it would be best to only work with the files that are definitely contained in the new data?

In [12]:
valid_ids = set(xml_ids).intersection(set(meta.id))
len(valid_ids)

5214

In [13]:
valid_xml_files = [f for f in all_xml_files if f.name.split(".")[0] in valid_ids]

In [14]:
# Trying to understand where the duplicates in the DataFrame come from,
# but apparently no duplicate files are present
seen = set()
duplicates = set()
for file in valid_xml_files:
    name = file.name.split(".")[0]
    if name in seen:
        print(f"{name} already seen!")
        duplicates.add(name)
    seen.add(name)
print("Duplicate IDs:", len(duplicates))

Duplicate IDs: 0


The following contains the parsing methods: The aim is to only collect narrative text, and thus figures, tables, their captions, references, formulas, and notes are discarded. Claude (GUI version, free tier) has been utilized in writing the following code.

In [ ]:
BODY_SKIP_ANCESTORS = {"figure", "figDesc", "table", "note", "abstract"}
ABSTRACT_SKIP_ANCESTORS = {"figure", "figDesc", "table", "note"}
SKIP_INLINE = {"ref", "formula", "note"}

WS_BEFORE_PUNCT = re.compile(r'\s+([.,;:!?])')
EXTRA_WS = re.compile(r'\s+')

def iter_paragraphs(container, skip_ancestors=frozenset()):
    """Yield (owning_div, paragraph) pairs. Each <p> is visited exactly once,
    owned by its nearest ancestor <div> (avoids double-counting nested divs)."""
    for p in container.find_all("p"):
        if p.find_parent(skip_ancestors):
            continue
        owning_div = p.find_parent("div")
        yield owning_div, p

def get_section_title(div):
    """Get title from <head> if present, else fall back to a bare leading
    text node (GROBID sometimes emits div titles as untagged text)."""
    head = div.find("head", recursive=False)
    if head:
        return head.get_text(strip=True)

    # Fallback: look for a leading NavigableString among direct children
    for child in div.contents:
        if isinstance(child, NavigableString):
            text = child.strip()
            if text:
                return text
        else:
            # Hit a real tag before finding loose text - no bare title here
            break

    return None

def extract_from_container(container, section_type, paper_id=None, skip_ancestors=frozenset()):
    if container is None:
        return []

    results = []
    title_cache = {}

    def title_for(div):
        if div is None:
            return None
        key = id(div)
        if key not in title_cache:
            title_cache[key] = get_section_title(div)
        return title_cache[key]

    # paragraph_index should be per-owning-div, so track per div
    para_counters = {}

    for owning_div, p in iter_paragraphs(container, skip_ancestors):
        key = id(owning_div)
        i = para_counters.get(key, 0)
        para_counters[key] = i + 1

        title = title_for(owning_div)

        sentences = p.find_all("s")
        units = sentences if sentences else [p]

        for j, unit in enumerate(units):
            unit_copy = copy.copy(unit)
            for tag in unit_copy.find_all(SKIP_INLINE):
                tag.decompose()

            text = unit_copy.get_text(separator=" ", strip=True)
            text = WS_BEFORE_PUNCT.sub(r'\1', text)
            text = EXTRA_WS.sub(' ', text).strip()

            if text:
                results.append({
                    "paper_id": paper_id,
                    "section_type": section_type,   # "body" or "abstract"
                    "section": title,
                    "paragraph_index": i,
                    "sentence_index": j,
                    "text": text
                })

    return results


def extract_sentences(soup, paper_id=None):
    body = soup.find("body")
    return extract_from_container(body, "body", paper_id, skip_ancestors=BODY_SKIP_ANCESTORS)


def extract_abstract_sentences(soup, paper_id=None):
    results = []
    for abstract in soup.find_all("abstract"):
        results.extend(extract_from_container(abstract, "abstract", paper_id, skip_ancestors=ABSTRACT_SKIP_ANCESTORS))
    return results


def extract_all_sentences(soup, paper_id=None):
    return extract_abstract_sentences(soup, paper_id) + extract_sentences(soup, paper_id)

In [16]:
def parse_sentences_from_xmls(filepaths):
    fulltext_data = []
    for i, xml_file in tqdm(enumerate(filepaths)):
        with gzip.open(xml_file, "rb") as file:
            paper_id = xml_file.name.split(".")[0]
            soup = BeautifulSoup(file, "xml")
            fulltext_data.extend(extract_all_sentences(soup, paper_id))

    print(f"{len(fulltext_data)} sentences extracted")
    return fulltext_data

In [22]:
def find_path_to_example(id_str):
    for f in valid_xml_files:
        if f.name.split(".")[0] == id_str:
            return f
    print("Path not found")

def print_example(id_str):
    example_file = find_path_to_example(id_str)
    if example_file:
        with gzip.open(example_file, "rb") as file:
            example_soup = BeautifulSoup(file, "xml")
        print(example_soup.prettify())

## Test the parsing method

In [17]:
test_data = parse_sentences_from_xmls(valid_xml_files[:100])
test_df = pd.DataFrame.from_records(test_data)
test_df.info()

100it [00:09, 10.32it/s]

28059 sentences extracted
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28059 entries, 0 to 28058
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   paper_id         28059 non-null  object
 1   section_type     28059 non-null  object
 2   section          26363 non-null  object
 3   paragraph_index  28059 non-null  int64 
 4   sentence_index   28059 non-null  int64 
 5   text             28059 non-null  object
dtypes: int64(2), object(4)
memory usage: 1.3+ MB


In [26]:
test_df.section_type.value_counts()

section_type
body        27342
abstract      717
Name: count, dtype: int64

In [30]:
test_df[test_df["section_type"] == "abstract"]["section"].unique()

array([None, 'Motivation:', 'Results:', 'Methods:', 'R E S U M O',
       "Owkin's team :", 'I. INTRODUCTION'], dtype=object)

In [33]:
test_df[test_df["section_type"] == "body"]["section"].value_counts()[lambda x: x > 100]

section
Introduction                                                             1364
Results                                                                   359
Discussion                                                                338
INTRODUCTION                                                              334
Methods                                                                   225
Conclusion                                                                214
Conclusions                                                               182
Related Work                                                              164
BEHIND THE BORDER TRADE BARRIERS: THE DOMESTIC REGULATORY ENVIRONMENT     146
I. INTRODUCTION                                                           135
A Combinatorial-Based Knowledge Production Function                       133
In-Sample Performance                                                     119
A. Perceptive Interpretability                          

## Parse sentences from all valid XML files

In [34]:
fulltext_data = parse_sentences_from_xmls(valid_xml_files)

5214it [08:02, 10.81it/s]

1152524 sentences extracted


In [35]:
fulltext_df = pd.DataFrame.from_records(fulltext_data)
fulltext_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1152524 entries, 0 to 1152523
Data columns (total 6 columns):
 #   Column           Non-Null Count    Dtype 
---  ------           --------------    ----- 
 0   paper_id         1152524 non-null  object
 1   section_type     1152524 non-null  object
 2   section          1093088 non-null  object
 3   paragraph_index  1152524 non-null  int64 
 4   sentence_index   1152524 non-null  int64 
 5   text             1152524 non-null  object
dtypes: int64(2), object(4)
memory usage: 52.8+ MB


In [36]:
fulltext_df = fulltext_df.astype({col: "category" for col in ["paper_id", "section_type"]})
fulltext_df

,paper_id,section_type,section,paragraph_index,sentence_index,text
0,W169306565,abstract,None,0,0,Artificial Intelligence (AI) has rapidly becom...
1,W169306565,abstract,None,0,1,"However, the growth of AI also raises concerns..."
2,W169306565,abstract,None,0,2,This research paper provides a comprehensive o...
3,W169306565,abstract,None,0,3,"To achieve this goal, a thorough review of exi..."
4,W169306565,abstract,None,0,4,"The literature review focused on the economic,..."
...,...,...,...,...,...,...
1152519,W7140044697,body,Conclusion,3,0,"Regarding RQ3, a gap analysis of the Ukrainian..."
1152520,W7140044697,body,Conclusion,3,1,"This finding supports an ""adaptive augmentatio..."
1152521,W7140044697,body,Conclusion,4,0,The integrative framework synthesises TPACK (w...
1152522,W7140044697,body,Conclusion,4,1,"Each pillar addresses a distinct question, and..."


In [37]:
fulltext_df.section_type.value_counts()

section_type
body        1112254
abstract      40270
Name: count, dtype: int64

In [38]:
fulltext_df[(fulltext_df["section_type"] == "abstract") & (~fulltext_df["section"].isna())]

,paper_id,section_type,section,paragraph_index,sentence_index,text
6091,W2804167233,abstract,Motivation:,0,0,The immune system has potential to present a w...
6092,W2804167233,abstract,Motivation:,0,1,This means of surveillances allows the immune ...
6093,W2804167233,abstract,Motivation:,0,2,"However, given the breadth of the epitope repe..."
6094,W2804167233,abstract,Results:,0,0,"We present Allele-Integrated MHC (AI-MHC), a d..."
6095,W2804167233,abstract,Results:,0,1,Our architecture utilizes a convolutional neur...
...,...,...,...,...,...,...
1149536,W4411128392,abstract,Methods,0,2,It then guided the model to produce a synthesi...
1149537,W4411128392,abstract,Methods,0,3,Details on the system prompt are provided in S...
1151249,W7128682674,abstract,Scientific Contribution,0,0,We present manual and AI-based methods to impr...
1151250,W7128682674,abstract,Scientific Contribution,0,1,These enhancements increase the interpretabili...


In [39]:
fulltext_df[(fulltext_df["section_type"] == "abstract")]["section"].nunique()

277

In [41]:
fulltext_df[fulltext_df.duplicated(keep=False)].sort_values(["paper_id", "text"])

,paper_id,section_type,section,paragraph_index,sentence_index,text
417669,W2236370645,body,Conclusions,0,1,However detecting or counting the species of i...
417679,W2236370645,body,Conclusions,0,1,However detecting or counting the species of i...
417670,W2236370645,body,Conclusions,0,2,"Other important issues are UAV regulations, op..."
417680,W2236370645,body,Conclusions,0,2,"Other important issues are UAV regulations, op..."
417672,W2236370645,body,Conclusions,1,1,The TMBM algorithm was able to detect the targ...
...,...,...,...,...,...,...
1034701,W7128053212,body,Stimuli and procedure,1,3,Each participant completed only one version.
857649,W7128295376,body,Predictor variables,0,0,Step 1
857661,W7128295376,body,Predictor variables,0,0,Step 1
857650,W7128295376,body,Predictor variables,1,0,Step 2


Still not fully sure why these exact duplicates exist in the first place, but going to get rid of them.

In [42]:
fulltext_df = fulltext_df.drop_duplicates(keep="first").reset_index(drop=True)

In [43]:
fulltext_df["text_len"] = fulltext_df["text"].str.len()

In [44]:
fulltext_df["num_tokens"] = fulltext_df["text"].str.count(" ") + 1

In [45]:
fulltext_df.isna().sum()

paper_id               0
section_type           0
section            59435
paragraph_index        0
sentence_index         0
text                   0
text_len               0
num_tokens             0
dtype: int64

In [46]:
fulltext_df.describe()

,paragraph_index,sentence_index,text_len,num_tokens
count,1.149210e+06,1.149210e+06,1.149210e+06,1.149210e+06
mean,2.383242e+00,3.242078e+00,1.458662e+02,2.225898e+01
std,3.881336e+00,4.463578e+00,8.461671e+01,1.550953e+01
min,0.000000e+00,0.000000e+00,1.000000e+00,1.000000e+00
25%,0.000000e+00,1.000000e+00,9.300000e+01,1.400000e+01
50%,1.000000e+00,2.000000e+00,1.350000e+02,2.100000e+01
75%,3.000000e+00,4.000000e+00,1.860000e+02,2.800000e+01
max,1.020000e+02,1.220000e+02,1.860200e+04,7.244000e+03


One could try to get rid of parsing errors, but it might be out of scope for the current timeline and goals.

In [47]:
fulltext_df[fulltext_df["num_tokens"] < 3]["text"].value_counts()[lambda x: x > 10]

text
.                          1339
).                          401
where                       375
1.                          343
"                           206
                           ... 
doi.org/10.32388/0386CK      11
3.1.2                        11
[..                          11
2.3.                         11
RQ3.                         11
Name: count, Length: 166, dtype: int64

The documents where the Grobid parse has failed (`[BAD_INPUT_DATA] PDF to XML conversion failed with error code: 1`) should be removed.

In [48]:
fulltext_df[(fulltext_df.duplicated("text")) & (fulltext_df["num_tokens"] > 2)]["text"].value_counts()[lambda x: x > 5].head(20)

text
[BAD_INPUT_DATA] PDF to XML conversion failed with error code: 1                                                                                                                                                                                                                                                                                                           236
The authors declare no conflict of interest.                                                                                                                                                                                                                                                                                                                               172
All rights reserved.                                                                                                                                                                                                                                                 

In [49]:
fulltext_df[fulltext_df["text"] == "[BAD_INPUT_DATA] PDF to XML conversion failed with error code: 1"]

,paper_id,section_type,section,paragraph_index,sentence_index,text,text_len,num_tokens
11015,W2965647350,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10
22792,W2613716520,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10
28732,W2998696444,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10
29933,W3031652686,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10
32356,W4220795697,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10
...,...,...,...,...,...,...,...,...
702620,W2898173783,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10
703321,W2932267232,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10
705336,W3118793301,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10
705380,W3126150727,body,None,0,0,[BAD_INPUT_DATA] PDF to XML conversion failed ...,64,10


In [50]:
failed_ids = set(fulltext_df[fulltext_df["text"] == "[BAD_INPUT_DATA] PDF to XML conversion failed with error code: 1"].paper_id)

In [51]:
fulltext_df = fulltext_df[~fulltext_df["paper_id"].isin(failed_ids)]

While the exact duplicates have been removed, there are still plenty of papers that include the same text multiple times in the same paper. Manual inspection with a few examples suggests that these are how the papers are written, not a bug, but I cannot claim this generalizes to all cases.

In [52]:
fulltext_df[(fulltext_df.duplicated(["paper_id", "text"], keep=False)) & (fulltext_df["num_tokens"] > 2)].sort_values(["paper_id", "text"])

,paper_id,section_type,section,paragraph_index,sentence_index,text,text_len,num_tokens
414451,W2117592456,body,Our Experiences,3,2,Students saw how simple search programs evolve...,167,23
414475,W2117592456,body,Conclusion,1,3,Students saw how simple search programs evolve...,167,23
414699,W2137486141,abstract,None,0,0,Information quality assurance under the existe...,277,39
414704,W2137486141,body,Introduction,0,0,Information quality assurance under the existe...,277,39
414876,W2137486141,body,Conclusions,0,0,Information quality assurance under the existe...,277,39
...,...,...,...,...,...,...,...,...
833153,W7134959100,body,B.2. Explaining encoded absences in a trained ...,1,0,The non-target attribution is computed through...,211,41
855129,W7136068103,abstract,None,0,2,"We present Manifold, a specification-driven or...",137,15
855151,W7136068103,body,Introduction,0,15,"We present Manifold, a specification-driven or...",137,15
855134,W7136068103,abstract,None,0,7,across all experiments.,23,3


In [53]:
print_example("W2137486141")

<?xml version="1.0" encoding="utf-8"?>
<TEI xml:space="preserve" xmlns="http://www.tei-c.org/ns/1.0" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xsi:schemaLocation="http://www.tei-c.org/ns/1.0 https://raw.githubusercontent.com/kermitt2/grobid/master/grobid-home/schemas/xsd/Grobid.xsd">
 <teiHeader xml:lang="en">
  <fileDesc>
   <titleStmt>
    <title level="a" type="main">
     Information Quality Assurance by Lazy Exploration of Information Source Combinations Space in Open Multi-Agent Systems
    </title>
    <funder ref="#_kB5tZPR">
     <orgName type="full">
      U.S. Department of Defense, Small Business Technology Transfer (STTR) Program
     </orgName>
    </funder>
    <funder>
     <orgName type="full">
      ScenPro, Inc
     </orgName>
    </funder>
   </titleStmt>
   <publicationStmt>
    <publisher/>
    <availability status="unknown">
     <licence/>
    </availability>
   </publicationStmt>
   <sourceDesc>
    <biblSt

In [54]:
fulltext_df = fulltext_df.merge(meta[["id", "year", "domain"]], left_on="paper_id", right_on="id").drop(columns="id")

In [55]:
fulltext_df = fulltext_df.astype({col: "category" for col in ["year", "domain"]})
fulltext_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1148973 entries, 0 to 1148972
Data columns (total 10 columns):
 #   Column           Non-Null Count    Dtype   
---  ------           --------------    -----   
 0   paper_id         1148973 non-null  object  
 1   section_type     1148973 non-null  category
 2   section          1089775 non-null  object  
 3   paragraph_index  1148973 non-null  int64   
 4   sentence_index   1148973 non-null  int64   
 5   text             1148973 non-null  object  
 6   text_len         1148973 non-null  int64   
 7   num_tokens       1148973 non-null  int64   
 8   year             1148973 non-null  category
 9   domain           1148973 non-null  category
dtypes: category(3), int64(4), object(3)
memory usage: 64.7+ MB


In [56]:
fulltext_df.describe(include="all")

,paper_id,section_type,section,paragraph_index,sentence_index,text,text_len,num_tokens,year,domain
count,1148973,1148973,1089775,1.148973e+06,1.148973e+06,1148973,1.148973e+06,1.148973e+06,1148973.0,1148973
unique,4971,2,55408,NaN,NaN,1123174,NaN,NaN,11.0,4
top,W2221189325,body,Introduction,NaN,NaN,.,NaN,NaN,2021.0,Physical Sciences
freq,3968,1108703,66281,NaN,NaN,1339,NaN,NaN,151391.0,407092
mean,NaN,NaN,NaN,2.383733e+00,3.242747e+00,NaN,1.458830e+02,2.226151e+01,NaN,NaN
std,NaN,NaN,NaN,3.881585e+00,4.463795e+00,NaN,8.461727e+01,1.551013e+01,NaN,NaN
min,NaN,NaN,NaN,0.000000e+00,0.000000e+00,NaN,1.000000e+00,1.000000e+00,NaN,NaN
25%,NaN,NaN,NaN,0.000000e+00,1.000000e+00,NaN,9.300000e+01,1.400000e+01,NaN,NaN
50%,NaN,NaN,NaN,1.000000e+00,2.000000e+00,NaN,1.350000e+02,2.100000e+01,NaN,NaN
75%,NaN,NaN,NaN,3.000000e+00,4.000000e+00,NaN,1.860000e+02,2.800000e+01,NaN,NaN


In [57]:
fulltext_df

,paper_id,section_type,section,paragraph_index,sentence_index,text,text_len,num_tokens,year,domain
0,W169306565,abstract,None,0,0,Artificial Intelligence (AI) has rapidly becom...,176,25,2023,Social Sciences
1,W169306565,abstract,None,0,1,"However, the growth of AI also raises concerns...",133,21,2023,Social Sciences
2,W169306565,abstract,None,0,2,This research paper provides a comprehensive o...,174,25,2023,Social Sciences
3,W169306565,abstract,None,0,3,"To achieve this goal, a thorough review of exi...",104,19,2023,Social Sciences
4,W169306565,abstract,None,0,4,"The literature review focused on the economic,...",148,22,2023,Social Sciences
...,...,...,...,...,...,...,...,...,...,...
1148968,W7140044697,body,Conclusion,3,0,"Regarding RQ3, a gap analysis of the Ukrainian...",215,30,2026,Social Sciences
1148969,W7140044697,body,Conclusion,3,1,"This finding supports an ""adaptive augmentatio...",103,12,2026,Social Sciences
1148970,W7140044697,body,Conclusion,4,0,The integrative framework synthesises TPACK (w...,202,27,2026,Social Sciences
1148971,W7140044697,body,Conclusion,4,1,"Each pillar addresses a distinct question, and...",96,12,2026,Social Sciences


In [71]:
(len(fulltext_df) // 3)

382991

In [58]:
fulltext_df.paper_id.nunique()

4971

In [62]:
for i, row in enumerate(fulltext_df.sample(50, random_state=987).itertuples(), start=1):
    print(f"{i:>3}) {row.text}")

  1) The close human management of workers is replaced by a digital system of verification that enables requesters to judge the quality of work and choose not to pay for poor-quality tasks.
  2) Notably, non-reasoning models can also scale test-time compute with traditional methods like majority voting, but those methods fail to close the performance gap with reasoning models, even when controlling for the total number of tokens generated.
  3) These are rich sources of data for researchers interested in analyzing how media coverage evolves as the virus spreads globally, or in tracking misleading reports and disinformation in the media.
  4) We conclude with three recommendations for a revised ALTAI.
  5) Patients who met the polysomnographic criteria of central sleep apnoea were not eligible for this study.
  6) This paper focuses on the goal of creating an open-domain conversational agent that can display many skills, and blend them in a seamless and engaging way.
  7) To guarantee t

In [63]:
fulltext_df.to_csv("/scratch/project_2017974/data_collection/data/parsed-fulltexts/fulltexts.csv", index=False)

In [72]:
fulltext_df.sample(20, random_state=999).to_csv("/scratch/project_2017974/data_collection/data/parsed-fulltexts/sample.csv", index=False)